<a href="https://colab.research.google.com/github/youssefabozaidyou/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

My baseline rule identifies content that may need refresh or optimization.

The rule uses two signals:

1. Staleness:
- Content that has not been updated for a long time may need a refresh.

2. CTR vs Position:
- Pages with good ranking positions but weak CTR may need snippet optimization.

Scoring logic:

- +50 points if days_since_last_update > 180 days.
- +40 points if avg_position <= 10 and CTR is below 0.05.
- +10 points if search volume is high.

Reason codes:

- STALE_CONTENT
- LOW_CTR_GOOD_POSITION
- HIGH_VOLUME

Actions:

- REFRESH_CONTENT
- OPTIMIZE_SNIPPET
- REVIEW

In [10]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1,30,90,180,10000],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "180+ days"
    ]
)

staleness_check = df.groupby(
    "staleness_bucket",
    observed=True
).agg(
    n=("content_id","count"),
    avg_ctr=("ctr","mean"),
    avg_position=("avg_position","mean")
)

staleness_check

,n,avg_ctr,avg_position
staleness_bucket,,,
0-30 days,20480,0.609021,15.685166
31-90 days,175,0.117543,16.538286
91-180 days,9171,0.238367,17.901461
180+ days,174,3.693276,11.325862


Verdict: CONFIRMED

Reason:
Observed that older content groups show different performance patterns,
which supports using freshness as a directional refresh signal.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import pandas as pd
import os


# Load dataset
df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)


def baseline_rule(row):

    score = 0
    reasons = []


    # Signal 1: Staleness
    if row["days_since_last_update"] > 180:
        score += 50
        reasons.append("STALE_CONTENT")


    # Signal 2: CTR vs Position
    if row["avg_position"] <= 10 and row["ctr"] < 0.05:
        score += 40
        reasons.append("LOW_CTR_GOOD_POSITION")


    # Volume signal
    if row["search_volume"] >= 100:
        score += 10
        reasons.append("HIGH_VOLUME")


    # Action
    if score >= 70:
        action = "REFRESH_CONTENT"

    elif score >= 40:
        action = "OPTIMIZE_SNIPPET"

    else:
        action = "MONITOR"


    return pd.Series(
        {
            "score": score,
            "reason_code": "|".join(reasons) if reasons else "NO_SIGNAL",
            "action": action
        }
    )


# Apply rule
scored_df = df.join(
    df.apply(baseline_rule, axis=1)
)


# Rank queue
ranked_queue = scored_df.sort_values(
    by="score",
    ascending=False
)


# Save output
os.makedirs(
    "work/outputs",
    exist_ok=True
)


ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


ranked_queue.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
23619,content_24abafed9707,client_8722616204,480.0,0.00,LOW,0.00,keyword article,transactional,3246.0,21393.0,...,0.00,50.00,0.0,low,top_3,down,-100.0,100,STALE_CONTENT|LOW_CTR_GOOD_POSITION|HIGH_VOLUME,REFRESH_CONTENT
15947,content_40e140ba2934,client_8722616204,720.0,0.29,LOW,0.78,keyword article,transactional,3306.0,21228.0,...,6.25,5.56,0.0,low,page_1,flat,NaN,100,STALE_CONTENT|LOW_CTR_GOOD_POSITION|HIGH_VOLUME,REFRESH_CONTENT
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.0,low,page_1,down,-95.6,100,STALE_CONTENT|LOW_CTR_GOOD_POSITION|HIGH_VOLUME,REFRESH_CONTENT
1339,content_1af4aceb4525,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,875.0,6016.0,...,0.00,0.00,0.0,low,top_3,new,NaN,90,STALE_CONTENT|LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
5003,content_17aa56bdad68,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,781.0,5651.0,...,0.00,0.00,0.0,low,page_1,stable,0.0,90,STALE_CONTENT|LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
10836,content_e748f498b262,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,1367.0,13108.0,...,0.00,100.00,0.0,low,top_3,new,NaN,90,STALE_CONTENT|LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
5833,content_6557f2b648e8,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3717.0,26454.0,...,0.00,100.00,0.0,low,page_1,up,166.7,90,STALE_CONTENT|LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
26054,content_a4da7c7eb188,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,746.0,6380.0,...,0.00,0.00,0.0,low,top_3,flat,NaN,90,STALE_CONTENT|LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
7719,content_f783292bc4a0,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,931.0,6547.0,...,0.00,100.00,0.0,low,page_1,flat,NaN,90,STALE_CONTENT|LOW_CTR_GOOD_POSITION,REFRESH_CONTENT
10998,content_b84c5db2dbad,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,829.0,5975.0,...,0.00,100.00,0.0,low,page_1,up,200.0,90,STALE_CONTENT|LOW_CTR_GOOD_POSITION,REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = ranked_queue.head(20)

top20[
    [
        "content_id",
        "action",
        "reason_code",
        "score",
        "days_since_last_update",
        "ctr",
        "avg_position"
    ]
]

,content_id,action,reason_code,score,days_since_last_update,ctr,avg_position
23619,content_24abafed9707,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION|HIGH_VOLUME,100,231,0.0,1.3
15947,content_40e140ba2934,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION|HIGH_VOLUME,100,231,0.0,4.5
21984,content_02b0d6e30129,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION|HIGH_VOLUME,100,313,0.0,6.9
1339,content_1af4aceb4525,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION,90,211,0.0,0.0
5003,content_17aa56bdad68,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION,90,211,0.0,4.0
10836,content_e748f498b262,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION,90,211,0.0,3.0
5833,content_6557f2b648e8,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION,90,183,0.0,9.2
26054,content_a4da7c7eb188,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION,90,211,0.0,3.0
7719,content_f783292bc4a0,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION,90,211,0.0,4.2
10998,content_b84c5db2dbad,REFRESH_CONTENT,STALE_CONTENT|LOW_CTR_GOOD_POSITION,90,211,0.0,7.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks + Leakage Check

Some picks may be weak because:
- Old content does not always mean poor performance.
- CTR can vary because of query intent or seasonality.

Leakage check:
- No labels were used.
- No future windows were used.
- Only observed content and performance signals were used.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.